# 03 — Models A, B, C

**Project:** *Selling Property Rental Ownership — A Hybrid Real Estate Model*  
**Author:** Dan Allouche · **Date:** May 2026

This notebook simulates the three ownership structures defined in the working paper and produces the headline risk-return chart that compares them on equal footing:

- **Model A** — classical full ownership of a single apartment.
- **Model B** — SAS share with pro-rata exposure to aggregate cash flows.
- **Model C** — CDO-style tranching (equity, mezzanine, senior).

The simulation is driven by the parameters fitted in `02_calibration.ipynb`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import yaml

from tranche_pricing.config import load_config
from tranche_pricing.pricing import compare_credit_models, runner as pricing_runner
from tranche_pricing.viz import figures
from tranche_pricing.viz.style import apply_style

apply_style()
pd.options.display.float_format = "{:,.4f}".format
cfg = load_config(ROOT / "config/paris_intermediate.yaml")
sim, gbm, vas, lgd, tranches, coupons = pricing_runner.build_inputs_from_yaml(cfg)
print(f"Scenario {cfg.scenario.name}: par={sim.par/1e6:.2f}M EUR, n_obligors={sim.n_obligors}, n_paths={sim.n_paths}")
print(f"GBM mu={gbm.mu:.4f} sigma={gbm.sigma:.4f}; Vasicek a={vas.a:.4f} b={vas.b:.4f} sigma_r={vas.sigma_r:.4f}")


## Run the comparison under the Gaussian copula baseline

In [ ]:
df, outputs = compare_credit_models(sim, gbm=gbm, vasicek=vas, lgd=lgd, tranches=tranches, coupons=coupons, models=("gaussian_copula",))
df_gauss = df[df["credit_model"] == "gaussian_copula"]
display(df_gauss.set_index("instrument")[["fair_price", "fair_to_par", "mean_ann_return", "risk_std", "risk_sharpe", "risk_var_95", "prob_negative_return"]])


## Figure #8 — Risk-adjusted metrics across instruments

In [ ]:
df_for_fig = df_gauss.assign(insurance="none")
fig = figures.fig_riskreturn_bars(results=df_for_fig, credit_model="gaussian_copula", insurance="none")
(ROOT / "artifacts/figures").mkdir(parents=True, exist_ok=True)
fig.savefig(ROOT / "artifacts/figures/fig_riskreturn_bars.pdf")
fig.savefig(ROOT / "artifacts/figures/fig_riskreturn_bars.png", dpi=200)


## Figure #9 — Risk-return frontier

In [ ]:
fig = figures.fig_pareto_frontier(results=df_for_fig)
fig.savefig(ROOT / "artifacts/figures/fig_pareto_frontier.pdf")
fig.savefig(ROOT / "artifacts/figures/fig_pareto_frontier.png", dpi=200)


---
**Next step:** `04_tranche_pricing.ipynb` repeats the exercise under the Student-t and Cox credit models to quantify how much the choice of model matters.